In [ ]:
+import numpy

In [ ]:
!pip install ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 22.0 MB/s eta 0:00:00


In [ ]:
import torch
torch.cuda.is_available()

False

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import io
import random
from pathlib  import Path
import requests
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms, models
from torchvision.models import ResNet18_Weights
print(torch.__version__)

2.10.0+cpu


In [ ]:
train_dir = Path(r"/content/drive/MyDrive/yolov8_train_car/Car_Demage_Severity/training")
test_dir = Path(r"/content/drive/MyDrive/yolov8_train_car/Car_Demage_Severity/validation")
save_dir = Path("/content/drive/MyDrive/yolov8_train_car/pkl")
save_dir.mkdir(parents=True, exist_ok=True)

model_path = save_dir / 'cnn_car.pkl'
seed=42
random.seed(seed)
torch.manual_seed(seed)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device


device(type='cpu')

In [ ]:
# Transforms
weights = ResNet18_Weights.DEFAULT
eval_transform = weights.transforms()

train_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomResizedCrop((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=10),
    transforms.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.15),
    transforms.ToTensor(),
    transforms.Normalize(mean=eval_transform.mean, std=eval_transform.std),
])

In [ ]:
# Datasets and loaders
full_train_for_train = datasets.ImageFolder(root=str(train_dir), transform=train_transform)
full_train_for_eval = datasets.ImageFolder(root=str(train_dir), transform=eval_transform)
test_dataset = datasets.ImageFolder(root=str(test_dir), transform=eval_transform)

indices = list(range(len(full_train_for_train)))
split = int(0.75 * len(indices))
g = torch.Generator().manual_seed(seed)
perm = torch.randperm(len(indices), generator=g).tolist()
train_idx, val_idx = perm[:split], perm[split:]

train_dataset = Subset(full_train_for_train, train_idx)
val_dataset = Subset(full_train_for_eval, val_idx)

batch_size = 32
pin_memory = torch.cuda.is_available()
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=0, pin_memory=pin_memory)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=0, pin_memory=pin_memory)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=0, pin_memory=pin_memory)

class_names = full_train_for_train.classes
print('Classes:', class_names)
print(f'Train/Val/Test: {len(train_dataset)}/{len(val_dataset)}/{len(test_dataset)}')

Classes: ['01-minor', '02-moderate', '03-severe']
Train/Val/Test: 2209/737/390


In [ ]:
@torch.no_grad()
def evaluate(model, loader, device):
    model.eval()
    total_loss = 0.0
    total_correct = 0
    total_samples = 0

    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        logits = model(images)
        loss = nn.functional.cross_entropy(logits, labels, reduction='sum')
        total_loss += loss.item()
        total_correct += (logits.argmax(dim=1) == labels).sum().item()
        total_samples += labels.size(0)

    return {
        'loss': total_loss / max(total_samples, 1),
        'accuracy': total_correct / max(total_samples, 1),
    }


def train_one_phase(model, train_loader, val_loader, device, epochs, optimizer, save_path):
    history = []
    best_val_acc = 0.0

    for epoch in range(1, epochs + 1):
        model.train()
        running_loss = 0.0
        running_correct = 0
        total_samples = 0

        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            logits = model(images)
            loss = nn.functional.cross_entropy(logits, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * labels.size(0)
            running_correct += (logits.argmax(dim=1) == labels).sum().item()
            total_samples += labels.size(0)

        train_metrics = {
            'loss': running_loss / max(total_samples, 1),
            'accuracy': running_correct / max(total_samples, 1),
        }
        val_metrics = evaluate(model, val_loader, device)

        if val_metrics['accuracy'] > best_val_acc:
            best_val_acc = val_metrics['accuracy']
            torch.save(model.state_dict(), save_path)

        history.append({'epoch': epoch, 'train': train_metrics, 'val': val_metrics})
        print(
            f"Epoch {epoch:02d} | train_loss={train_metrics['loss']:.4f} train_acc={train_metrics['accuracy']:.4f} | "
            f"val_loss={val_metrics['loss']:.4f} val_acc={val_metrics['accuracy']:.4f}"
        )

    print(f'Best val_acc: {best_val_acc:.4f}')
    return history

In [ ]:


# Build transfer learning model
model = models.resnet18(weights=weights).to(device)
num_ftrs = model.fc.in_features
model.fc = nn.Linear(num_ftrs, 3).to(device)

# Phase 1: freeze backbone, train only classifier head
for name, param in model.named_parameters():
    param.requires_grad = name.startswith('fc.')

optimizer_phase1 = torch.optim.Adam(model.fc.parameters(), lr=1e-3)
print('Phase 1: train classifier head')
history_phase1 = train_one_phase(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    device=device,
    epochs=50,
    optimizer=optimizer_phase1,
    save_path=model_path,
)

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 243MB/s]


Phase 1: train classifier head
Epoch 01 | train_loss=0.9089 train_acc=0.5885 | val_loss=0.7331 val_acc=0.6676
Epoch 02 | train_loss=0.7685 train_acc=0.6519 | val_loss=0.6815 val_acc=0.6893
Epoch 03 | train_loss=0.7301 train_acc=0.6795 | val_loss=0.6262 val_acc=0.7246
Epoch 04 | train_loss=0.7074 train_acc=0.6849 | val_loss=0.6014 val_acc=0.7503
Epoch 05 | train_loss=0.6750 train_acc=0.7157 | val_loss=0.6934 val_acc=0.6934
Epoch 06 | train_loss=0.6735 train_acc=0.7148 | val_loss=0.6342 val_acc=0.7151
Epoch 07 | train_loss=0.6694 train_acc=0.7112 | val_loss=0.5788 val_acc=0.7558
Epoch 08 | train_loss=0.6710 train_acc=0.7121 | val_loss=0.5633 val_acc=0.7693
Epoch 09 | train_loss=0.6785 train_acc=0.7057 | val_loss=0.5886 val_acc=0.7463
Epoch 10 | train_loss=0.6726 train_acc=0.7098 | val_loss=0.6630 val_acc=0.7083
Epoch 11 | train_loss=0.6660 train_acc=0.7121 | val_loss=0.5623 val_acc=0.7734
Epoch 12 | train_loss=0.6655 train_acc=0.7125 | val_loss=0.5799 val_acc=0.7558
Epoch 13 | train_loss

In [ ]:
# Phase 2: fine-tune layer4 + fc
for param in model.parameters():
    param.requires_grad = False
for param in model.layer4.parameters():
    param.requires_grad = True
for param in model.fc.parameters():
    param.requires_grad = True

optimizer_phase2 = torch.optim.Adam([
    {'params': model.layer4.parameters(), 'lr': 1e-4},
    {'params': model.fc.parameters(), 'lr': 5e-4},
])
print('Phase 2: fine-tune layer4 + fc')
history_phase2 = train_one_phase(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    device=device,
    epochs=50,
    optimizer=optimizer_phase2,
    save_path=model_path,
)

# Load best checkpoint and evaluate test
model.load_state_dict(torch.load(model_path, map_location=device))
test_metrics = evaluate(model, test_loader, device)
print('Best checkpoint test metrics:', test_metrics)
print('Saved model:', model_path)

Phase 2: fine-tune layer4 + fc
Epoch 01 | train_loss=0.6682 train_acc=0.7347 | val_loss=0.5270 val_acc=0.7992
Epoch 02 | train_loss=0.6006 train_acc=0.7565 | val_loss=0.5221 val_acc=0.8005
Epoch 03 | train_loss=0.5416 train_acc=0.7687 | val_loss=0.5319 val_acc=0.8114
Epoch 04 | train_loss=0.5249 train_acc=0.7859 | val_loss=0.5610 val_acc=0.7992
Epoch 05 | train_loss=0.4685 train_acc=0.8094 | val_loss=0.6014 val_acc=0.8005
Epoch 06 | train_loss=0.4564 train_acc=0.8189 | val_loss=0.5882 val_acc=0.8114
Epoch 07 | train_loss=0.4217 train_acc=0.8284 | val_loss=0.5443 val_acc=0.8100
Epoch 08 | train_loss=0.4128 train_acc=0.8289 | val_loss=0.5924 val_acc=0.7992
Epoch 09 | train_loss=0.3698 train_acc=0.8511 | val_loss=0.6403 val_acc=0.7897
Epoch 10 | train_loss=0.3691 train_acc=0.8538 | val_loss=0.6517 val_acc=0.8073
Epoch 11 | train_loss=0.3676 train_acc=0.8624 | val_loss=0.6640 val_acc=0.7788
Epoch 12 | train_loss=0.3586 train_acc=0.8597 | val_loss=0.6503 val_acc=0.8005
Epoch 13 | train_loss

In [ ]:
# Predict from image URL
def predict_from_url(url: str):
    resp = requests.get(url, timeout=20)
    resp.raise_for_status()

    image = Image.open(io.BytesIO(resp.content)).convert('RGB')
    x = eval_transform(image).unsqueeze(0).to(device)

    model.eval()
    with torch.no_grad():
        logits = model(x)
        probs = torch.softmax(logits, dim=1)[0].cpu()

    idx = int(torch.argmax(probs).item())
    return {
        'label': class_names[idx],
        'confidence': float(probs[idx].item()),
        'probs': {class_names[i]: float(probs[i].item()) for i in range(len(class_names))},
    }

In [ ]:
# Example URL prediction
import cv2
sample_url = 'https://tamanhhospital.vn/wp-content/uploads/2024/06/hinh-anh-nhan-biet-kien-ba-khoang.jpg'
result = predict_from_url(sample_url)
print(result)

{'label': '03-severe', 'confidence': 0.6306317448616028, 'probs': {'01-minor': 0.06459297239780426, '02-moderate': 0.30477529764175415, '03-severe': 0.6306317448616028}}
